# Remote Orchestration

High-level function `run_remote()` that orchestrates running `llm_runner`
operations on Isambard: setup, data transfer, SBATCH submission, polling,
result download.

Jobs are **content-addressed and idempotent**. Every job is identified by a
deterministic hash of (operation + config + input data). Identical requests
reuse cached results or attach to in-progress jobs.

In [ ]:
#|default_exp orchestrate

In [ ]:
#|export
import asyncio
import hashlib
import json
import tempfile
import uuid
from enum import Enum
from pathlib import Path, PurePosixPath
from typing import Any

import numpy as np

from functools import partial

from isambard_utils.config import IsambardConfig
from isambard_utils.ssh import arun as async_ssh_run, _get_config, _run_sync

_fprint = partial(print, flush=True)

In [ ]:
#|export
class TransferMode(str, Enum):
    """How to transfer input data to/from Isambard."""
    DIRECT = "direct"           # Tar + SSH pipe, no remote persistence
    UPLOAD = "upload"            # rsync to content-hashed folder, idempotent
    COMPRESSED = "compressed"   # tar.gz + SSH pipe to content-hashed folder, idempotent

## Job hash computation

In [ ]:
#|export
_ORCHESTRATION_ONLY_KEYS = frozenset({
    "transfer_modes", "output_transfer", "setup", "time", "job_name",
    "required_models", "isambard_config", "print_fn", "cache", "upload_timeout",
    "gpus",
})

def compute_job_hash(operation: str, inputs: dict[str, Any], config_dict: dict) -> str:
    """Compute a deterministic content hash for a remote job.

    SHA256 over operation name, canonical config JSON, and per-input value
    hashes (sorted by key). Computed locally, in-memory (no SSH, no disk I/O).

    Type normalisation: numpy arrays containing strings/objects are converted
    to lists before hashing, so np.array(["a","b"]) and ["a","b"] produce the
    same hash. Numeric arrays stay as numpy (their binary representation matters).

    Args:
        operation: Operation name (e.g. "embed", "llm_generate", "cosine_topk").
        inputs: Named inputs for the operation.
        config_dict: Operation config dict. Orchestration-only keys are excluded.

    Returns:
        Hex string of the SHA256 digest.

    Raises:
        TypeError: If an input value is not a supported type.
    """
    h = hashlib.sha256()

    # 1. Operation name
    h.update(operation.encode())

    # 2. Canonical config JSON (excluding orchestration-only keys)
    filtered_config = {k: v for k, v in config_dict.items()
                       if k not in _ORCHESTRATION_ONLY_KEYS}
    h.update(json.dumps(filtered_config, sort_keys=True, separators=(',', ':')).encode())

    # 3. Per-input value hashes (sorted by key)
    for key in sorted(inputs.keys()):
        value = inputs[key]
        h.update(key.encode())
        h.update(_hash_value(value))

    return h.hexdigest()

In [ ]:
#|export
def _hash_value(value: Any) -> bytes:
    """Compute a hash contribution for a single input value."""
    if isinstance(value, np.ndarray):
        # String/object arrays -> normalize to list for consistent hashing
        if value.dtype.kind in ('U', 'S', 'O'):
            return _hash_value(value.tolist())
        # Numeric arrays: hash dtype + shape + raw bytes
        vh = hashlib.sha256()
        vh.update(value.dtype.str.encode())
        vh.update(str(value.shape).encode())
        vh.update(value.tobytes())
        return vh.digest()
    if _is_json_hashable(value):
        return hashlib.sha256(
            json.dumps(value, sort_keys=True, separators=(',', ':')).encode()
        ).digest()
    raise TypeError(
        f"Cannot hash value of type {type(value).__name__}. "
        f"Supported types: numpy.ndarray, and JSON-serializable "
        f"(list, dict, str, int, float, bool, None)."
    )

In [ ]:
#|export
def _is_json_hashable(value) -> bool:
    """Check if a value can be JSON-serialized for hashing."""
    if isinstance(value, (str, int, float, bool, type(None))):
        return True
    if isinstance(value, (list, tuple)):
        return all(_is_json_hashable(item) for item in value)
    if isinstance(value, dict):
        return all(
            isinstance(k, str) and _is_json_hashable(v)
            for k, v in value.items()
        )
    return False

## In-process concurrency

In [ ]:
#|export
_JOB_LOCKS: dict[str, asyncio.Lock] = {}
_JOB_LOCKS_LOOP: asyncio.AbstractEventLoop | None = None

def _get_job_lock(job_hash: str) -> asyncio.Lock:
    """Get or create an asyncio.Lock for a given job hash.

    Invalidates all cached locks when the event loop changes (e.g. when
    running multiple pipeline runs sequentially via asyncio.run or in
    different thread pool workers).
    """
    global _JOB_LOCKS_LOOP
    loop = asyncio.get_running_loop()
    if _JOB_LOCKS_LOOP is not loop:
        _JOB_LOCKS.clear()
        _JOB_LOCKS_LOOP = loop
    if job_hash not in _JOB_LOCKS:
        _JOB_LOCKS[job_hash] = asyncio.Lock()
    return _JOB_LOCKS[job_hash]

## Remote status management

In [ ]:
#|export
async def _read_remote_status(cache_path: str, *, config: IsambardConfig) -> dict | None:
    """Read _status.json from a remote cache directory. Returns None if not found."""
    result = await async_ssh_run(f"cat {cache_path}/_status.json", config=config, check=False)
    if result.returncode != 0:
        return None
    try:
        return json.loads(result.stdout)
    except json.JSONDecodeError:
        return None

async def _write_remote_status(cache_path: str, status_dict: dict, *, config: IsambardConfig) -> None:
    """Write _status.json to a remote cache directory."""
    from isambard_utils.transfer import aupload_bytes
    await async_ssh_run(f"mkdir -p {cache_path}", config=config)
    await aupload_bytes(
        json.dumps(status_dict, indent=2).encode(),
        f"{cache_path}/_status.json",
        config=config,
    )

In [ ]:
#|export
async def _check_job_alive(job_id: str, *, config: IsambardConfig) -> str:
    """Check if a Slurm job is alive via squeue then sacct. Returns effective state."""
    from isambard_utils.slurm import ajob_state
    return await ajob_state(job_id, config=config)

## Upload, submit, download helpers

In [ ]:
#|export
async def _upload_inputs(
    cache_path: str,
    inputs: dict[str, Any],
    transfer_modes: dict[str, TransferMode],
    *,
    config: IsambardConfig,
    print_fn=_fprint,
) -> dict[str, str]:
    """Serialize and upload inputs to {cache_path}/inputs/{key}/. Returns manifest dict."""
    from isambard_utils.transfer import (
        aupload_tar_pipe, aupload_idempotent, aupload_compressed,
        compute_content_hash,
    )
    from llm_runner.serialization import serialize

    manifest = {}
    for key, value in inputs.items():
        mode = transfer_modes.get(key, TransferMode.DIRECT)
        with tempfile.TemporaryDirectory() as tmp:
            local_dir = Path(tmp) / key
            serialize({key: value}, local_dir)

            input_remote = f"{cache_path}/inputs/{key}"
            if mode == TransferMode.DIRECT:
                await aupload_tar_pipe(str(local_dir), input_remote, config=config)
            elif mode == TransferMode.UPLOAD:
                content_hash = compute_content_hash(str(local_dir))
                input_remote = await aupload_idempotent(
                    str(local_dir), f"{cache_path}/inputs", content_hash, config=config,
                )
            elif mode == TransferMode.COMPRESSED:
                from isambard_utils.transfer import _aupload_compressed_direct
                await _aupload_compressed_direct(str(local_dir), input_remote, config=config)
            manifest[key] = input_remote

    return manifest

In [ ]:
#|export
async def _submit_job(
    cache_path: str,
    operation: str,
    config_dict: dict,
    manifest: dict[str, str],
    *,
    slurm_job_name: str,
    time: str,
    gpus: int = 1,
    config: IsambardConfig,
    print_fn=_fprint,
) -> "SlurmJob":
    """Write manifest and submit SBATCH job. Returns SlurmJob."""
    from isambard_utils.transfer import aupload_bytes
    from isambard_utils.slurm import asubmit, SlurmJob
    from isambard_utils.sbatch import generate as generate_sbatch, SbatchConfig

    outputs_remote = f"{cache_path}/outputs"
    await async_ssh_run(f"mkdir -p {outputs_remote}", config=config)

    # Write manifest
    manifest_remote = f"{cache_path}/manifest.json"
    await aupload_bytes(json.dumps(manifest).encode(), manifest_remote, config=config)

    # Write job metadata
    meta = {"operation": operation, "config_dict": config_dict, "slurm_job_name": slurm_job_name}
    await aupload_bytes(
        json.dumps(meta, indent=2).encode(),
        f"{cache_path}/_job_meta.json",
        config=config,
    )

    # Generate and submit SBATCH
    filtered_config = {k: v for k, v in config_dict.items()
                       if k not in _ORCHESTRATION_ONLY_KEYS}
    config_json = json.dumps(filtered_config)
    sbatch_cfg = SbatchConfig(
        job_name=slurm_job_name,
        time=time,
        gpus=gpus,
        env_vars={"PYTHONPATH": f"{config.project_dir}/src"},
        command=(
            f"python -m llm_runner {operation} "
            f"--manifest {manifest_remote} "
            f"--outputs-dir {outputs_remote} "
            f"--config '{config_json}'"
        ),
    )
    sbatch_script = generate_sbatch(sbatch_cfg, isambard_config=config)
    print_fn(f"run_remote [{slurm_job_name}]: submitting job...")
    job = await asubmit(sbatch_script, config=config)
    print_fn(f"run_remote [{slurm_job_name}]: submitted job {job.job_id}")
    return job

In [ ]:
#|export
async def _poll_job(job_id: str, *, label: str, cache_path: str,
                    config: IsambardConfig, print_fn=_fprint) -> dict:
    """Poll a Slurm job until it finishes, return sacct summary."""
    from isambard_utils.slurm import await_job, ajob_log

    def _on_poll(status):
        if status:
            print_fn(f"run_remote [{label}]: job {job_id} state={status.get('state', '?')}")

    final = await await_job(job_id, config=config, on_poll=_on_poll)
    final_state = final.get("state", "UNKNOWN")
    print_fn(f"run_remote [{label}]: job finished with state={final_state}")

    if final_state != "COMPLETED":
        stdout_log = await ajob_log(job_id, config=config, stream="stdout", tail=50)
        stderr_log = await ajob_log(job_id, config=config, stream="stderr", tail=50)
        # Also read the runner's own status.json which has the Python traceback
        runner_status = ""
        rs = await async_ssh_run(
            f"cat {cache_path}/outputs/status.json", config=config, check=False,
        )
        if rs.returncode == 0:
            runner_status = rs.stdout
        raise RuntimeError(
            f"run_remote job {label} (id={job_id}) failed with state={final_state}.\n"
            f"cache: {cache_path}\n"
            f"--- runner status ---\n{runner_status}\n"
            f"--- stdout (last 50 lines) ---\n{stdout_log}\n"
            f"--- stderr (last 50 lines) ---\n{stderr_log}"
        )
    return final

In [ ]:
#|export
async def _download_outputs(
    cache_path: str,
    output_transfer: TransferMode,
    *,
    config: IsambardConfig,
) -> dict[str, Any]:
    """Download and deserialize outputs from remote cache."""
    from isambard_utils.transfer import adownload_tar_pipe, adownload
    from llm_runner.serialization import deserialize

    outputs_remote = f"{cache_path}/outputs"

    # Check runner status.json
    check = await async_ssh_run(f"cat {outputs_remote}/status.json", config=config, check=False)
    if check.returncode == 0:
        runner_status = json.loads(check.stdout)
        if runner_status.get("state") != "COMPLETED":
            raise RuntimeError(
                f"run_remote runner reported non-completed state: {runner_status}"
            )

    with tempfile.TemporaryDirectory() as tmp:
        local_outputs = Path(tmp) / "outputs"
        local_outputs.mkdir()

        if output_transfer == TransferMode.DIRECT:
            await adownload_tar_pipe(outputs_remote, str(local_outputs), config=config)
        else:
            await adownload(outputs_remote + "/", str(local_outputs), config=config)

        return deserialize(local_outputs)

## Public status/cache functions

In [ ]:
#|export
async def ajob_status(job_hash: str, *, config: IsambardConfig | None = None) -> dict | None:
    """Read the status of a cached job, verifying against Slurm state if submitted.

    Returns a dict with state, job_id, slurm_job_name, hash, etc., or None if
    no cache entry exists for this hash.
    """
    config = _get_config(config)
    cache_path = str(PurePosixPath(config.project_dir) / ".runner_cache" / job_hash)
    status = await _read_remote_status(cache_path, config=config)
    if status is None:
        return None

    # If submitted, verify Slurm state
    if status.get("state") == "submitted" and status.get("job_id"):
        slurm_state = await _check_job_alive(status["job_id"], config=config)
        if slurm_state in ("COMPLETED",):
            status["state"] = "completed"
        elif slurm_state in ("FAILED", "CANCELLED", "TIMEOUT", "NODE_FAIL",
                              "OUT_OF_MEMORY", "PREEMPTED"):
            status["state"] = "failed"
            status["slurm_final_state"] = slurm_state

    status["hash"] = job_hash
    return status

def job_status(job_hash: str, *, config: IsambardConfig | None = None) -> dict | None:
    """Read the status of a cached job. See ajob_status()."""
    return _run_sync(ajob_status(job_hash, config=config))

In [ ]:
#|export
async def aclear_job_cache(job_hash: str, *, config: IsambardConfig | None = None) -> None:
    """Remove a cached job directory from the remote."""
    config = _get_config(config)
    cache_path = str(PurePosixPath(config.project_dir) / ".runner_cache" / job_hash)
    await async_ssh_run(f"rm -rf {cache_path}", config=config, check=False)
    _JOB_LOCKS.pop(job_hash, None)

def clear_job_cache(job_hash: str, *, config: IsambardConfig | None = None) -> None:
    """Remove a cached job directory from the remote. See aclear_job_cache()."""
    _run_sync(aclear_job_cache(job_hash, config=config))

## Runner setup

In [ ]:
#|export
_setup_done = False
_setup_lock: asyncio.Lock | None = None
_setup_lock_loop: asyncio.AbstractEventLoop | None = None

def _get_setup_lock() -> asyncio.Lock:
    """Get the setup lock, recreating it if the event loop has changed."""
    global _setup_lock, _setup_lock_loop, _setup_done
    loop = asyncio.get_running_loop()
    if _setup_lock_loop is not loop:
        _setup_lock = asyncio.Lock()
        _setup_lock_loop = loop
        _setup_done = False
    return _setup_lock

async def asetup_runner(*, config: IsambardConfig | None = None, print_fn=_fprint,
                        force: bool = False) -> None:
    """Ensure llm_runner is deployed on Isambard (idempotent, targeted).

    Uploads a minimal remote_pyproject.toml (llm_runner deps only) and
    src/llm_runner/. Creates dirs, installs uv if needed, and runs uv sync.

    Skips entirely if already called successfully in this process, unless
    force=True. Uses a lock to prevent concurrent setup attempts.

    Args:
        config: Isambard configuration.
        print_fn: Print function for progress logging.
        force: Run setup even if already done this session.
    """
    global _setup_done
    async with _get_setup_lock():
        if _setup_done and not force:
            return
        import subprocess
        config = _get_config(config)
        from isambard_utils.env import _aensure_uv, _aensure_venv, _aensure_cuda_torch, _afix_lustre_hardlinks
        from isambard_utils.transfer import aupload as async_rsync_upload, aupload_bytes
        import llm_runner

        try:
            # 1. Create remote dirs
            await async_ssh_run(
                f"mkdir -p {config.project_dir}/src {config.hf_cache_dir} {config.logs_dir}",
                config=config, retries=3, print_fn=print_fn,
            )

            # 2. Ensure uv
            await _aensure_uv(config=config)

            # 3. Upload remote_pyproject.toml as pyproject.toml
            runner_src = Path(llm_runner.__path__[0])
            remote_pyproject = runner_src / "assets" / "remote_pyproject.toml"
            await aupload_bytes(remote_pyproject.read_bytes(),
                                f"{config.project_dir}/pyproject.toml", config=config)

            # 4. Rsync llm_runner source
            print_fn("runner setup: syncing llm_runner source...")
            await async_rsync_upload(
                str(runner_src) + "/", f"{config.project_dir}/src/llm_runner",
                config=config, exclude=["__pycache__", "*.pyc"],
            )

            # 5. Install deps
            await _aensure_venv(config=config, print_fn=print_fn)

            # 6. Ensure CUDA torch (PyPI installs CPU-only on ARM64)
            print_fn("runner setup: ensuring CUDA torch...")
            await _aensure_cuda_torch(config=config)

            # 7. Fix Lustre hardlinks (one-time migration to copy mode)
            await _afix_lustre_hardlinks(config=config)
            print_fn("runner setup: done")
            _setup_done = True

        except subprocess.CalledProcessError as e:
            raise RuntimeError(
                f"Runner setup failed (exit {e.returncode}).\n"
                f"--- command ---\n{e.cmd}\n"
                f"--- stderr ---\n{e.stderr or '(empty)'}"
            ) from e

In [ ]:
#|export
def setup_runner(*, config: IsambardConfig | None = None, print_fn=_fprint,
                 force: bool = False) -> None:
    """Ensure llm_runner is deployed on Isambard (sync wrapper).

    See asetup_runner() for details.
    """
    _run_sync(asetup_runner(config=config, print_fn=print_fn, force=force))

## Main entry point: `arun_remote`

In [ ]:
#|export
async def arun_remote(
    operation: str,
    inputs: dict[str, Any],
    config_dict: dict,
    *,
    setup: bool = True,
    transfer_modes: dict[str, TransferMode] | TransferMode = TransferMode.DIRECT,
    output_transfer: TransferMode = TransferMode.DIRECT,
    job_name: str,
    time: str = "02:00:00",
    gpus: int = 1,
    required_models: list[str] | None = None,
    isambard_config: IsambardConfig | None = None,
    print_fn=_fprint,
    cache: bool = True,
    upload_timeout: int = 600,
) -> dict[str, Any]:
    """Run a llm_runner operation remotely on Isambard via SBATCH (async).

    When cache=True (default), jobs are content-addressed and idempotent:
    - Identical requests reuse cached results
    - Concurrent identical requests attach to the in-progress job
    - Failed jobs are automatically resubmitted

    Args:
        operation: Operation name ("embed", "llm_generate", "cosine_topk").
        inputs: Named inputs for the operation.
        config_dict: Operation config dict (model_name, dtype, etc.).
        setup: Run runner setup before job (default True).
        transfer_modes: Transfer mode per input key, or a single mode for all.
        output_transfer: Transfer mode for downloading outputs.
        job_name: Base Slurm job name.
        time: Slurm time limit (e.g. "02:00:00").
        gpus: Number of GPUs to request (default 1).
        required_models: HuggingFace model names to pre-cache.
        isambard_config: Isambard configuration.
        print_fn: Print function for progress logging.
        cache: Use content-addressed caching (default True). If False, uses
            a UUID-based temp directory (old behavior).
        upload_timeout: Seconds to wait when another process is uploading
            before treating as stale (default 600).

    Returns:
        Dict of operation outputs (deserialized).
    """
    from isambard_utils.models import aensure_model

    ic = _get_config(isambard_config)

    # Normalize transfer_modes to per-key dict
    if isinstance(transfer_modes, TransferMode):
        transfer_modes = {key: transfer_modes for key in inputs}

    # 1. Setup runner
    if setup:
        print_fn(f"run_remote [{job_name}]: setting up runner...")
        await asetup_runner(config=ic, print_fn=print_fn)

    # 2. Pre-cache models
    for model_name in (required_models or []):
        print_fn(f"run_remote [{job_name}]: ensuring model {model_name}...")
        await aensure_model(model_name, config=ic, print_fn=print_fn)

    if not cache:
        return await _run_remote_uncached(
            operation, inputs, config_dict,
            transfer_modes=transfer_modes,
            output_transfer=output_transfer,
            job_name=job_name, time=time, gpus=gpus,
            config=ic, print_fn=print_fn,
        )

    # Content-addressed idempotent path
    job_hash = compute_job_hash(operation, inputs, config_dict)
    cache_path = str(PurePosixPath(ic.project_dir) / ".runner_cache" / job_hash)
    slurm_job_name = f"{job_name}_{job_hash[:8]}"
    print_fn(f"run_remote [{job_name}]: hash={job_hash}")
    print_fn(f"run_remote [{job_name}]: cache={cache_path}")
    lock = _get_job_lock(job_hash)

    async with lock:
        return await _run_remote_cached(
            operation, inputs, config_dict,
            job_hash=job_hash, cache_path=cache_path,
            slurm_job_name=slurm_job_name,
            transfer_modes=transfer_modes,
            output_transfer=output_transfer,
            time=time, gpus=gpus, upload_timeout=upload_timeout,
            config=ic, print_fn=print_fn,
        )

In [ ]:
#|export
def _extract_accounting(sacct: dict) -> dict:
    """Extract accounting fields from a sacct result dict for storage/return.

    Returns a dict with the timing and resource fields, or empty dict if
    no accounting data is available.
    """
    fields = ("elapsed_seconds", "start_time", "end_time",
              "allocated_cpus", "allocated_gpus", "node_hours")
    return {k: sacct[k] for k in fields if k in sacct}


async def _run_remote_cached(
    operation: str,
    inputs: dict[str, Any],
    config_dict: dict,
    *,
    job_hash: str,
    cache_path: str,
    slurm_job_name: str,
    transfer_modes: dict[str, TransferMode],
    output_transfer: TransferMode,
    time: str,
    gpus: int = 1,
    upload_timeout: int,
    config: IsambardConfig,
    print_fn=_fprint,
) -> dict[str, Any]:
    """Idempotency state machine for content-addressed jobs."""
    import time as time_mod

    status = await _read_remote_status(cache_path, config=config)

    if status is None:
        # Cache miss: upload + submit
        print_fn(f"run_remote [{slurm_job_name}]: cache miss, uploading inputs...")
        await _write_remote_status(cache_path, {"state": "uploading"}, config=config)

        manifest = await _upload_inputs(
            cache_path, inputs, transfer_modes,
            config=config, print_fn=print_fn,
        )
        job = await _submit_job(
            cache_path, operation, config_dict, manifest,
            slurm_job_name=slurm_job_name, time=time, gpus=gpus,
            config=config, print_fn=print_fn,
        )
        await _write_remote_status(cache_path, {
            "state": "submitted", "job_id": job.job_id,
            "slurm_job_name": slurm_job_name,
        }, config=config)

        sacct = await _poll_job(job.job_id, label=slurm_job_name, cache_path=cache_path, config=config, print_fn=print_fn)
        accounting = _extract_accounting(sacct)
        await _write_remote_status(cache_path, {
            "state": "completed", "job_id": job.job_id,
            "slurm_job_name": slurm_job_name,
            "slurm_accounting": accounting,
        }, config=config)

        print_fn(f"run_remote [{slurm_job_name}]: downloading outputs...")
        result = await _download_outputs(cache_path, output_transfer, config=config)
        if accounting:
            result["_slurm_accounting"] = accounting
        print_fn(f"run_remote [{slurm_job_name}]: done")
        return result

    state = status.get("state", "")

    if state == "completed":
        # Cache hit: download from cache
        print_fn(f"run_remote [{slurm_job_name}]: cache hit (completed), downloading outputs...")
        result = await _download_outputs(cache_path, output_transfer, config=config)
        cached_accounting = status.get("slurm_accounting")
        if cached_accounting:
            result["_slurm_accounting"] = cached_accounting
        print_fn(f"run_remote [{slurm_job_name}]: done (from cache)")
        return result

    if state == "submitted":
        job_id = status.get("job_id")
        if job_id:
            slurm_state = await _check_job_alive(job_id, config=config)
            if slurm_state in ("PENDING", "RUNNING"):
                # Attach to running job
                print_fn(f"run_remote [{slurm_job_name}]: attaching to running job {job_id}...")
                sacct = await _poll_job(job_id, label=slurm_job_name, cache_path=cache_path, config=config, print_fn=print_fn)
                accounting = _extract_accounting(sacct)
                await _write_remote_status(cache_path, {
                    "state": "completed", "job_id": job_id,
                    "slurm_job_name": slurm_job_name,
                    "slurm_accounting": accounting,
                }, config=config)
                print_fn(f"run_remote [{slurm_job_name}]: downloading outputs...")
                result = await _download_outputs(cache_path, output_transfer, config=config)
                if accounting:
                    result["_slurm_accounting"] = accounting
                print_fn(f"run_remote [{slurm_job_name}]: done (attached)")
                return result
            elif slurm_state == "COMPLETED":
                # Job completed but status wasn't updated -- query sacct for accounting
                from isambard_utils.slurm import _asacct_status
                sacct = await _asacct_status(job_id, config=config)
                accounting = _extract_accounting(sacct)
                await _write_remote_status(cache_path, {
                    "state": "completed", "job_id": job_id,
                    "slurm_job_name": slurm_job_name,
                    "slurm_accounting": accounting,
                }, config=config)
                print_fn(f"run_remote [{slurm_job_name}]: job already completed, downloading...")
                result = await _download_outputs(cache_path, output_transfer, config=config)
                if accounting:
                    result["_slurm_accounting"] = accounting
                print_fn(f"run_remote [{slurm_job_name}]: done (from cache)")
                return result
            else:
                # Stale: job dead, resubmit
                print_fn(f"run_remote [{slurm_job_name}]: stale job {job_id} (state={slurm_state}), resubmitting...")
                await _write_remote_status(cache_path, {"state": "failed", "job_id": job_id}, config=config)
                # Fall through to failed handling

        # If no job_id or we fell through from stale
        status = await _read_remote_status(cache_path, config=config)
        state = status.get("state", "") if status else ""

    if state == "uploading":
        # Wait for another process to finish uploading
        print_fn(f"run_remote [{slurm_job_name}]: another process is uploading, waiting...")
        start = time_mod.time()
        while True:
            await asyncio.sleep(10)
            elapsed = time_mod.time() - start
            status = await _read_remote_status(cache_path, config=config)
            if status is None or status.get("state") != "uploading":
                break
            if elapsed > upload_timeout:
                print_fn(f"run_remote [{slurm_job_name}]: upload timeout ({upload_timeout}s), treating as stale")
                await _write_remote_status(cache_path, {"state": "failed"}, config=config)
                break

        # Re-enter state machine with updated status
        return await _run_remote_cached(
            operation, inputs, config_dict,
            job_hash=job_hash, cache_path=cache_path,
            slurm_job_name=slurm_job_name,
            transfer_modes=transfer_modes,
            output_transfer=output_transfer,
            time=time, gpus=gpus, upload_timeout=upload_timeout,
            config=config, print_fn=print_fn,
        )

    if state == "failed":
        # Resubmit: check if inputs exist, skip upload if they do
        print_fn(f"run_remote [{slurm_job_name}]: previous job failed, resubmitting...")
        check = await async_ssh_run(
            f"test -d {cache_path}/inputs", config=config, check=False,
        )
        if check.returncode == 0:
            # Inputs exist, read manifest
            manifest_check = await async_ssh_run(
                f"cat {cache_path}/manifest.json", config=config, check=False,
            )
            if manifest_check.returncode == 0:
                manifest = json.loads(manifest_check.stdout)
            else:
                # No manifest, re-upload
                await _write_remote_status(cache_path, {"state": "uploading"}, config=config)
                manifest = await _upload_inputs(
                    cache_path, inputs, transfer_modes,
                    config=config, print_fn=print_fn,
                )
        else:
            # No inputs, upload from scratch
            await _write_remote_status(cache_path, {"state": "uploading"}, config=config)
            manifest = await _upload_inputs(
                cache_path, inputs, transfer_modes,
                config=config, print_fn=print_fn,
            )

        job = await _submit_job(
            cache_path, operation, config_dict, manifest,
            slurm_job_name=slurm_job_name, time=time, gpus=gpus,
            config=config, print_fn=print_fn,
        )
        await _write_remote_status(cache_path, {
            "state": "submitted", "job_id": job.job_id,
            "slurm_job_name": slurm_job_name,
        }, config=config)

        sacct = await _poll_job(job.job_id, label=slurm_job_name, cache_path=cache_path, config=config, print_fn=print_fn)
        accounting = _extract_accounting(sacct)
        await _write_remote_status(cache_path, {
            "state": "completed", "job_id": job.job_id,
            "slurm_job_name": slurm_job_name,
            "slurm_accounting": accounting,
        }, config=config)

        print_fn(f"run_remote [{slurm_job_name}]: downloading outputs...")
        result = await _download_outputs(cache_path, output_transfer, config=config)
        if accounting:
            result["_slurm_accounting"] = accounting
        print_fn(f"run_remote [{slurm_job_name}]: done (resubmitted)")
        return result

    raise RuntimeError(f"Unexpected cache state {state!r} for job hash {job_hash}")

In [ ]:
#|export
async def _run_remote_uncached(
    operation: str,
    inputs: dict[str, Any],
    config_dict: dict,
    *,
    transfer_modes: dict[str, TransferMode],
    output_transfer: TransferMode,
    job_name: str,
    time: str,
    gpus: int = 1,
    config: IsambardConfig,
    print_fn=_fprint,
) -> dict[str, Any]:
    """Run a job without caching, using a UUID-based temp dir (backward compat)."""
    from isambard_utils.transfer import (
        aupload_tar_pipe, adownload_tar_pipe,
        aupload_idempotent, aupload_compressed,
        aupload_bytes, compute_content_hash,
    )
    from isambard_utils.slurm import asubmit, await_job, ajob_log
    from isambard_utils.sbatch import generate as generate_sbatch, SbatchConfig
    from llm_runner.serialization import serialize, deserialize

    work_dir = f"{config.project_dir}/.runner_jobs/{job_name}_{uuid.uuid4().hex[:8]}"
    outputs_remote = f"{work_dir}/outputs"

    # Transfer inputs
    print_fn(f"run_remote [{job_name}]: transferring inputs...")
    await async_ssh_run(f"mkdir -p {work_dir} {outputs_remote}", config=config)
    manifest = {}

    for key, value in inputs.items():
        mode = transfer_modes.get(key, TransferMode.DIRECT)
        with tempfile.TemporaryDirectory() as tmp:
            local_dir = Path(tmp) / key
            serialize({key: value}, local_dir)

            if mode == TransferMode.DIRECT:
                remote_dir = f"{work_dir}/inputs/{key}"
                await aupload_tar_pipe(str(local_dir), remote_dir, config=config)
                manifest[key] = remote_dir
            elif mode == TransferMode.UPLOAD:
                cache_dir = str(PurePosixPath(config.project_dir) / ".runner_cache")
                content_hash = compute_content_hash(str(local_dir))
                remote_dir = await aupload_idempotent(
                    str(local_dir), cache_dir, content_hash, config=config,
                )
                manifest[key] = remote_dir
            elif mode == TransferMode.COMPRESSED:
                cache_dir = str(PurePosixPath(config.project_dir) / ".runner_cache")
                content_hash = compute_content_hash(str(local_dir))
                remote_dir = await aupload_compressed(
                    str(local_dir), cache_dir, content_hash, config=config,
                )
                manifest[key] = remote_dir

    # Write manifest
    manifest_remote = f"{work_dir}/manifest.json"
    await aupload_bytes(json.dumps(manifest).encode(), manifest_remote, config=config)

    # Submit SBATCH
    config_json = json.dumps(config_dict)
    sbatch_cfg = SbatchConfig(
        job_name=job_name,
        time=time,
        gpus=gpus,
        env_vars={"PYTHONPATH": f"{config.project_dir}/src"},
        command=(
            f"python -m llm_runner {operation} "
            f"--manifest {manifest_remote} "
            f"--outputs-dir {outputs_remote} "
            f"--config '{config_json}'"
        ),
    )
    sbatch_script = generate_sbatch(sbatch_cfg, isambard_config=config)
    print_fn(f"run_remote [{job_name}]: submitting job...")
    job = await asubmit(sbatch_script, config=config)
    print_fn(f"run_remote [{job_name}]: submitted job {job.job_id}")

    # Poll
    sacct = await _poll_job(job.job_id, label=job_name, cache_path=work_dir, config=config, print_fn=print_fn)
    accounting = _extract_accounting(sacct)

    # Check status.json
    check = await async_ssh_run(f"cat {outputs_remote}/status.json", config=config, check=False)
    if check.returncode == 0:
        runner_status = json.loads(check.stdout)
        if runner_status.get("state") != "COMPLETED":
            raise RuntimeError(
                f"run_remote runner for {job_name} reported: {runner_status}"
            )

    # Download outputs
    print_fn(f"run_remote [{job_name}]: downloading outputs...")
    with tempfile.TemporaryDirectory() as tmp:
        local_outputs = Path(tmp) / "outputs"
        local_outputs.mkdir()

        if output_transfer == TransferMode.DIRECT:
            await adownload_tar_pipe(outputs_remote, str(local_outputs), config=config)
        else:
            from isambard_utils.transfer import adownload
            await adownload(outputs_remote + "/", str(local_outputs), config=config)

        result = deserialize(local_outputs)

    # Cleanup work dir
    await async_ssh_run(f"rm -rf {work_dir}", config=config, check=False)
    if accounting:
        result["_slurm_accounting"] = accounting
    print_fn(f"run_remote [{job_name}]: done")

    return result

In [ ]:
#|export
def run_remote(
    operation: str,
    inputs: dict[str, Any],
    config_dict: dict,
    *,
    setup: bool = True,
    transfer_modes: dict[str, TransferMode] | TransferMode = TransferMode.DIRECT,
    output_transfer: TransferMode = TransferMode.DIRECT,
    job_name: str,
    time: str = "02:00:00",
    gpus: int = 1,
    required_models: list[str] | None = None,
    isambard_config: IsambardConfig | None = None,
    print_fn=_fprint,
    cache: bool = True,
    upload_timeout: int = 600,
) -> dict[str, Any]:
    """Run a llm_runner operation remotely on Isambard via SBATCH.

    See arun_remote() for full documentation.
    """
    return _run_sync(arun_remote(
        operation, inputs, config_dict,
        setup=setup,
        transfer_modes=transfer_modes,
        output_transfer=output_transfer,
        job_name=job_name, time=time, gpus=gpus,
        required_models=required_models,
        isambard_config=isambard_config,
        print_fn=print_fn,
        cache=cache,
        upload_timeout=upload_timeout,
    ))